In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Hallucination_RAG_Project')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu sentence-transformers groq -q

import os
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
from getpass import getpass

# 1. Setup API and Models
if "GROQ_API_KEY" not in os.environ:
    print("Please enter your Groq API Key:")
    os.environ["GROQ_API_KEY"] = getpass()
client = Groq()

print("Loading data and embedding model...")
df = pd.read_csv("./data/prototype_qa_data.csv")
knowledge_base = df['knowledge'].dropna().tolist()
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Building FAISS index...")
document_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True, normalize_embeddings=True)
index = faiss.IndexFlatIP(document_embeddings.shape[1])
index.add(document_embeddings)

# 2. STEP 8: Build the Basic Context Filter (System C Prototype)
def retrieve_with_basic_filter(query, k=3, similarity_threshold=0.5):
    """
    Retrieves Top-K chunks, but only keeps them if their cosine similarity
    score is strictly greater than the threshold.
    """
    query_embedding = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(query_embedding, k)

    selected_chunks = []
    # distances[0] contains the similarity scores for the retrieved chunks
    for score, idx in zip(distances[0], indices[0]):
        if idx != -1 and score >= similarity_threshold:
            selected_chunks.append({
                "text": knowledge_base[idx],
                "score": score
            })

    return selected_chunks

# 3. Build SYSTEM C
def system_c_basic_semantic_rag(question, k=3, threshold=0.5):
    """
    Proposed Basic RAG: Filters out irrelevant context before generation.
    """
    filtered_results = retrieve_with_basic_filter(question, k, threshold)

    if not filtered_results:
        return "System C Filter: No highly relevant context found to answer this question confidently.", []

    # Extract just the text for the LLM prompt
    selected_contexts = [res["text"] for res in filtered_results]
    context_block = "\n\n".join(selected_contexts)

    prompt = f"""You are a helpful AI assistant. Answer the user's question based strictly on the provided context. If the context does not contain the answer, say "I cannot answer based on the provided context."

CONTEXT:
{context_block}

QUESTION:
{question}
"""
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=200
        )
        return response.choices[0].message.content.strip(), filtered_results
    except Exception as e:
        return f"Error: {str(e)}", []

# 4. Test System C
test_question = "Which magazine was started first Arthur's Magazine or First for Women?"
print(f"\nQUESTION: {test_question}")

system_c_answer, evidence = system_c_basic_semantic_rag(test_question, k=3, threshold=0.4)

print("\n--- SYSTEM C (Basic Proposed RAG) OUTPUT ---")
print(f"SELECTED EVIDENCE ({len(evidence)} chunks passed the filter):")
for i, ev in enumerate(evidence):
    print(f"  [Chunk {i+1} | Score: {ev['score']:.4f}] {ev['text']}")
print(f"\nANSWER:\n{system_c_answer}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.8 MB/s eta 0:00:00
Please enter your Groq API Key:
··········
Loading data and embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index...

QUESTION: Which magazine was started first Arthur's Magazine or First for Women?

--- SYSTEM C (Basic Proposed RAG) OUTPUT ---
SELECTED EVIDENCE (3 chunks passed the filter):
  [Chunk 1 | Score: 0.8751] Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.
  [Chunk 2 | Score: 0.4361] Jane was an American magazine created to appeal to the women who grew up reading "Sassy Magazine"; Jane Pratt was the founding editor of each.First for Women is a woman's magazine published by Bauer Media Group in the USA.
  [Chunk 3 | Score: 0.4030] Sojourners magazine is a progressive monthly publication of the American Christian social justice organization Sojourners, which arose out of the Sojourners Community.KO Magazine was a popular United States boxing magazine.

ANSWER:
I can answer that. 

Arthur's Magazine was published in 1844, and F